# Formative 2: Customer Data Merge, EDA and Product Recommendation
**Contributor: Tifare**  |  Source: two assignment Google Sheets  
This notebook is reproducible and calls the accompanying implementation in `src/pipeline.py`. It does not implement face or voice authentication.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from pipeline import prepare_data, make_eda, train_model, FEATURES
import pandas as pd
from IPython.display import display, Image

## 1. Load and inspect source files
The social IDs use an `A` prefix while transaction IDs use numeric characters. Multiple social observations per customer must be aggregated to avoid multiplying transaction records on join.

In [ ]:
social_raw = pd.read_csv(ROOT / "data/raw/customer_social_profiles.csv")
transaction_raw = pd.read_csv(ROOT / "data/raw/customer_transactions.csv")
print("Social data:", social_raw.shape, "Transactions:", transaction_raw.shape)
display(social_raw.head(), transaction_raw.head())
display(social_raw.isna().sum().rename("Social nulls"), transaction_raw.isna().sum().rename("Transaction nulls"))

## 2. Clean, aggregate, and merge
Standardize IDs, fix types, remove duplicate social records, reduce repeated observations to one profile per customer, merge with transaction-left join, and validate the result.

In [ ]:
social, transactions, merged, trainable, summary = prepare_data()
print(json.dumps({k:v for k,v in summary.items() if k not in ["numeric_summary", "nulls_social", "nulls_transactions"]}, indent=2))
assert len(merged) == len(transactions)
display(merged.head(), trainable[FEATURES+["product_category"]].head())

## 3. Exploratory data analysis
Five labeled charts: product distribution, amount boxplots, numerical correlations, engagement histogram, and test confusion matrix. Purchase amount and customer rating are used only for descriptive analysis, not as pre-purchase input features.

In [ ]:
make_eda(trainable)
print("Numeric summary statistics:")
display(trainable[["engagement_mean", "interest_mean", "social_activity_count", "purchase_amount", "customer_rating"]].describe())
for name in ["01_product_distribution.png", "02_purchase_amount_boxplot.png", "03_correlation_matrix.png", "04_engagement_histogram.png"]:
    display(Image(filename=str(ROOT / "figures" / name), width=730))

## 4. Fit and evaluate the product category classifier
Split by **customer ID** to avoid train-test contamination when users have repeated transactions. Compare majority-class baseline, logistic regression and random forest with grouped CV. Select the best CV macro-F1 on training data; evaluate once on the held-out customers.

In [ ]:
evaluation = train_model(trainable, summary)
print("Selected model:", evaluation["selected_model"])
print("Test accuracy:", round(evaluation["test_accuracy"], 3))
print("Test macro F1:", round(evaluation["test_f1_macro"], 3))
print("Test log loss:", round(evaluation["test_log_loss"], 3))
display(pd.DataFrame(evaluation["model_selection_cv"]).T)
display(Image(filename=str(ROOT / "figures/05_confusion_matrix.png"), width=680))

## 5. Findings, caveats, and integration
The models show limited predictive power on these small data. The dataset has only product **categories**, not individual products. Source social records have no timestamps to prove their scores were available at the purchase moment. Results are exploratory. For the full demo, use the saved trained recommendation model **only after** the separate face and voice verification modules have approved a user.

In [ ]:
from joblib import load
bundle = load(ROOT / "models/product_recommender.joblib")
example = trainable[FEATURES].iloc[[0]]
print("Example (known profile) category prediction:", bundle["model"].predict(example)[0])
print("Saved outputs:", sorted(p.name for p in (ROOT / "data/processed").iterdir()))